# Note 1: Naive Gaussian Elimination

**Goal:** Solve $Ax = b$ for a dense matrix $A$ using the simplest possible direct method.

## The Problem

Given an $n \times n$ matrix $A$ and an $n$-vector $b$, find $x$ such that $Ax = b$.

This is the fundamental problem at the heart of nearly all numerical optimization. Every Newton step, every interior point iteration, every least-squares fit reduces to solving a linear system.

## The Idea

Gaussian elimination transforms $Ax = b$ into an upper triangular system $Ux = c$ by systematically eliminating entries below the diagonal. Then we solve by **back substitution**.

### Forward Elimination

For each column $k = 0, 1, \ldots, n-2$:
- For each row $i = k+1, \ldots, n-1$:
  - Compute the multiplier $m_{ik} = a_{ik} / a_{kk}$
  - Subtract $m_{ik}$ times row $k$ from row $i$

The entry $a_{kk}$ used for division is called the **pivot**. If it's zero, naive GE fails.

### Back Substitution

Once we have the upper triangular system $Ux = c$:

$$x_i = \frac{1}{u_{ii}}\left(c_i - \sum_{j=i+1}^{n-1} u_{ij} x_j\right), \quad i = n-1, n-2, \ldots, 0$$

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

In [ ]:
def naive_gauss(A, b):
    """Solve Ax = b by naive Gaussian elimination (no pivoting)."""
    n = len(b)
    # Work on copies — augmented matrix [A | b]
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    
    # Forward elimination
    for k in range(n - 1):
        if abs(A[k, k]) < 1e-15:
            raise ValueError(f"Zero pivot at column {k}!")
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]       # multiplier
            A[i, k+1:] -= m * A[k, k+1:] # update row i
            b[i] -= m * b[k]              # update RHS
            A[i, k] = 0.0                 # (zeroed by construction)
    
    # Back substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
    
    return x

## Example 1: A Well-Behaved System

Let's start with a simple $3 \times 3$ system where naive GE works perfectly.

In [ ]:
A = np.array([
    [2.0, 1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0, 1.0, 2.0]
])
b = np.array([8.0, -11.0, -3.0])

x = naive_gauss(A, b)
print(f"Solution: x = {x}")
print(f"Residual: ||Ax - b|| = {np.linalg.norm(A @ x - b):.2e}")

The exact solution is $x = [2, 3, -1]$. Let's verify.

In [ ]:
print(f"Expected: [2, 3, -1]")
print(f"Got:      {x}")
print(f"Max error: {np.max(np.abs(x - [2, 3, -1])):.2e}")

## Where Naive GE Fails

### Failure Mode 1: Zero Pivot

If $a_{kk} = 0$ during elimination, we divide by zero.

In [ ]:
A_bad = np.array([
    [0.0, 1.0],
    [1.0, 1.0]
])
b_bad = np.array([1.0, 2.0])

try:
    naive_gauss(A_bad, b_bad)
except ValueError as e:
    print(f"Failed: {e}")

# But the system is perfectly solvable! Just swap the rows:
print(f"\nNumPy solves it fine: x = {np.linalg.solve(A_bad, b_bad)}")

### Failure Mode 2: Numerical Instability

Even when pivots aren't zero, **small pivots** cause catastrophic loss of precision. This is the more insidious problem.

Consider a system where the first pivot is tiny ($\epsilon \approx 10^{-15}$):

In [ ]:
eps = 1e-15
A_ill = np.array([
    [eps, 1.0],
    [1.0, 1.0]
])
b_ill = np.array([1.0, 2.0])

x_naive = naive_gauss(A_ill, b_ill)
x_numpy = np.linalg.solve(A_ill, b_ill)

print(f"Naive GE:  x = {x_naive}")
print(f"NumPy:     x = {x_numpy}")
print(f"\nNaive residual:  {np.linalg.norm(A_ill @ x_naive - b_ill):.2e}")
print(f"NumPy residual:  {np.linalg.norm(A_ill @ x_numpy - b_ill):.2e}")

The multiplier $m = 1/\epsilon \approx 10^{15}$ amplifies rounding errors catastrophically. The naive solution can be completely wrong.

## Cost Analysis

Gaussian elimination costs:
- **Forward elimination:** $\frac{2}{3}n^3 + O(n^2)$ floating-point operations
- **Back substitution:** $n^2$ operations
- **Total:** $O(n^3)$

Let's verify this empirically:

In [ ]:
import time

sizes = [50, 100, 200, 400]
times = []

for n in sizes:
    A = np.random.randn(n, n)
    A = A + n * np.eye(n)  # make diagonally dominant so naive GE works
    b = np.random.randn(n)
    
    t0 = time.perf_counter()
    x = naive_gauss(A, b)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    
    residual = np.linalg.norm(A @ x - b) / np.linalg.norm(b)
    print(f"n={n:4d}  time={elapsed:.4f}s  relative residual={residual:.2e}")

# Check cubic scaling
for i in range(1, len(sizes)):
    ratio = times[i] / times[i-1]
    size_ratio = sizes[i] / sizes[i-1]
    print(f"n: {sizes[i-1]} -> {sizes[i]}  time ratio: {ratio:.1f}x  (cubic predicts {size_ratio**3:.1f}x)")

## What We've Learned

1. Gaussian elimination transforms $Ax = b$ into $Ux = c$ via row operations, then solves by back substitution
2. It costs $O(n^3)$ — this is **unavoidable** for dense matrices
3. It fails on zero pivots and is unstable with small pivots
4. We need **pivoting** to fix both problems

## What's Next

In Note 2, we add **partial pivoting** — swapping rows to always use the largest available pivot. This gives us the standard LU factorization $PA = LU$ that production solvers (LAPACK, NumPy) use.

---

*This is Note 1 in a series building from basic Gaussian elimination to the multifrontal sparse solver used in [ripopt](https://github.com/jkitchin/ripopt).*